In [17]:
import os, sys, time, json, requests
import pandas as pd

In [2]:
import requests

url = "https://wft-geo-db.p.rapidapi.com/v1/geo/cities"

headers = {
	"x-rapidapi-key": "8fae97fb71mshb5bb9db6016c308p1347b9jsnf4256b706adb",
	"x-rapidapi-host": "wft-geo-db.p.rapidapi.com"
}

response = requests.get(url, headers=headers).json()

data = response['data']

In [3]:
data

[{'id': 3350606,
  'wikiDataId': 'Q24668',
  'type': 'CITY',
  'city': 'Aixirivall',
  'name': 'Aixirivall',
  'country': 'Andorra',
  'countryCode': 'AD',
  'region': 'Sant Julià de Lòria',
  'regionCode': '06',
  'regionWdId': 'Q24282',
  'latitude': 42.46245,
  'longitude': 1.50209,
  'population': 1025},
 {'id': 3216144,
  'wikiDataId': 'Q24656',
  'type': 'CITY',
  'city': 'Aixovall',
  'name': 'Aixovall',
  'country': 'Andorra',
  'countryCode': 'AD',
  'region': 'Sant Julià de Lòria',
  'regionCode': '06',
  'regionWdId': 'Q24282',
  'latitude': 42.47635833,
  'longitude': 1.48949167,
  'population': 69},
 {'id': 3406038,
  'wikiDataId': 'Q4699394',
  'type': 'CITY',
  'city': 'Aixàs',
  'name': 'Aixàs',
  'country': 'Andorra',
  'countryCode': 'AD',
  'region': 'Sant Julià de Lòria',
  'regionCode': '06',
  'regionWdId': 'Q24282',
  'latitude': 42.48638889,
  'longitude': 1.46722222,
  'population': 0},
 {'id': 397,
  'wikiDataId': 'Q1863',
  'type': 'CITY',
  'city': 'Andorra 

In [4]:
response['links'][1]

{'rel': 'next', 'href': '/v1/geo/cities?offset=5&limit=5'}

In [5]:
url = "https://wft-geo-db.p.rapidapi.com/v1/geo/cities?offset=5&limit=5"

headers = {
	"x-rapidapi-key": "8fae97fb71mshb5bb9db6016c308p1347b9jsnf4256b706adb",
	"x-rapidapi-host": "wft-geo-db.p.rapidapi.com"
}

response = requests.get(url, headers=headers).json()

data = response['data']
print(type(data))

<class 'list'>


In [6]:
# Now actual process 
base_url = 'https://wft-geo-db.p.rapidapi.com' 
next_url = response['links'][1]['href']

count = response['metadata']['totalCount']
limit = 5

total_pages = (count + limit - 1) // limit

print(f"Total records: {count:,}")
print(f"Total pages: {total_pages:,}")

cites = pd.DataFrame()

Total records: 744,360
Total pages: 148,872


In [7]:
count

744360

In [8]:
all_data = []

for i in tqdm(range(1, total_pages+1), desc="Fetching cities", unit="page"):
    page = base_url + next_url

    response = requests.get(page, headers=headers)
    response.raise_for_status()
    response = response.json()

    data = response['data']
    all_data.append(pd.DataFrame(data))

    for link in response['links']:
        if link['rel'] == 'next':
            next_url = link['href']

    time.sleep(1)

cities = pd.concat(all_data, ignore_index=True)

print(f"Completed! Total records collected: {len(cities):,}")

Fetching cities:   0%|                                                       | 17/148872 [00:44<107:11:59,  2.59s/page]


KeyboardInterrupt: 

In [ ]:
cites

In [14]:
def show_progress(offset, total, start_time, start_offset, status=""):
    done = offset - start_offset
    elapsed = time.time() - start_time
    rate = done / elapsed if elapsed > 0 else 0
    remaining = (total - offset) / rate if rate > 0 else 0
    h, rem = divmod(int(remaining), 3600)
    m, s = divmod(rem, 60)
    pct = offset / total * 100
    sys.stdout.write(
        f"\r{offset:,}/{total:,} ({pct:.2f}%) | {rate:.1f} rec/s | ETA {h}h {m:02d}m {s:02d}s {status:<25}"
    )
    sys.stdout.flush()


In [15]:
def get_page(offset, max_retries=8):
    """Fetch one page with backoff on 429/5xx/network errors."""
    wait = 5
    for attempt in range(max_retries):
        try:
            r = session.get(base_url, params={"limit": LIMIT, "offset": offset}, timeout=30)
        except requests.RequestException:
            time.sleep(wait); wait = min(wait * 2, 300); continue

        if r.status_code == 200:
            return r.json()
        if r.status_code == 429 or r.status_code >= 500:
            retry_after = r.headers.get("Retry-After")
            sleep_for = int(retry_after) if retry_after and retry_after.isdigit() else wait
            if sleep_for > 3600:
                raise RuntimeError(f"Quota likely exhausted (Retry-After={sleep_for}s).")
            time.sleep(sleep_for)
            wait = min(wait * 2, 300)
            continue
        r.raise_for_status()
    raise RuntimeError(f"Gave up at offset {offset} after {max_retries} retries.")

In [18]:
base_url = "https://wft-geo-db.p.rapidapi.com/v1/geo/cities"
LIMIT = 10                 
CHECKPOINT = "cities_checkpoint.jsonl"
STATE = "state.json"

session = requests.Session()
session.headers.update(headers)   # your RapidAPI headers

# --- resume support ---
offset = 0
if os.path.exists(STATE):
    offset = json.load(open(STATE))["offset"]

first = get_page(offset)
total = first["metadata"]["totalCount"]
print(f"Total records: {total:,} | resuming at offset {offset:,}")

start_time, start_offset, last = time.time(), offset, 0.0
pending = first["data"]          # reuse the first response, no wasted call

try:
    with open(CHECKPOINT, "a", encoding="utf-8") as out:
        while offset < total:
            if pending is None:
                wait = MIN_INTERVAL - (time.monotonic() - last)
                if wait > 0:
                    time.sleep(wait)
                last = time.monotonic()
                data = get_page(offset)["data"]
            else:
                data, pending = pending, None
                last = time.monotonic()

            if not data:
                break

            out.write("\n".join(json.dumps(row) for row in data) + "\n")
            out.flush()

            offset += LIMIT
            with open(STATE, "w") as f:
                json.dump({"offset": offset}, f)

            show_progress(min(offset, total), total, start_time, start_offset)

except (KeyboardInterrupt, RuntimeError) as e:
    print(f"\nStopped: {e}\nResume later from offset {offset:,}.")

print()

Total records: 744,360 | resuming at offset 10
230/744,360 (0.03%) | 1.9 rec/s | ETA 106h 30m 02s                          
Stopped: 
Resume later from offset 230.



In [20]:
import os, sys, time, json, threading, requests
from concurrent.futures import ThreadPoolExecutor, as_completed

base_url = "https://wft-geo-db.p.rapidapi.com/v1/geo/cities"
LIMIT = 10
WORKERS = 4
REQ_PER_SEC = 1.0              # raise only if you see no 429s
OUT = "cities_pages.jsonl"     # one line per page: {"offset":..., "data":[...]}

session = requests.Session()
session.headers.update(headers)
session.mount("https://", requests.adapters.HTTPAdapter(pool_connections=WORKERS, pool_maxsize=WORKERS))

lock = threading.Lock()
next_slot = 0.0
pause_until = 0.0

def throttle():
    global next_slot
    with lock:
        start = max(time.monotonic(), next_slot, pause_until)
        next_slot = start + 1 / REQ_PER_SEC
    time.sleep(max(0, start - time.monotonic()))

def fetch(offset, max_retries=10):
    global pause_until
    for _ in range(max_retries):
        throttle()
        try:
            r = session.get(base_url, params={"limit": LIMIT, "offset": offset}, timeout=30)
        except requests.RequestException:
            time.sleep(5); continue
        if r.status_code == 200:
            return offset, r.json()
        if r.status_code == 429 or r.status_code >= 500:
            ra = r.headers.get("Retry-After")
            sleep_for = int(ra) if ra and ra.isdigit() else 30
            if sleep_for > 3600:
                raise RuntimeError("Daily quota likely exhausted.")
            with lock:
                pause_until = max(pause_until, time.monotonic() + sleep_for)
            continue
        r.raise_for_status()
    raise RuntimeError(f"Gave up at offset {offset}")

# --- resume: find offsets already saved ---
done = set()
if os.path.exists(OUT):
    with open(OUT, encoding="utf-8") as f:
        for line in f:
            try:
                done.add(json.loads(line)["offset"])
            except Exception:
                pass            # ignore a half-written last line

_, first = fetch(0)
total = first["metadata"]["totalCount"]
todo = [o for o in range(0, total, LIMIT) if o not in done]
print(f"Total: {total:,} | pages done: {len(done):,} | pages left: {len(todo):,}")

start, saved = time.time(), 0
pool = ThreadPoolExecutor(WORKERS)
try:
    futures = [pool.submit(fetch, o) for o in todo]
    with open(OUT, "a", encoding="utf-8") as out:
        for fut in as_completed(futures):
            offset, js = fut.result()
            out.write(json.dumps({"offset": offset, "data": js["data"]}) + "\n")
            out.flush()
            saved += 1
            rate = saved * LIMIT / (time.time() - start)
            eta = (len(todo) - saved) * LIMIT / rate if rate else 0
            sys.stdout.write(f"\r{saved:,}/{len(todo):,} pages | {rate:.1f} rec/s | ETA {eta/3600:.1f}h   ")
            sys.stdout.flush()
except (KeyboardInterrupt, RuntimeError) as e:
    print(f"\nStopped: {e}. Rerun to resume.")
finally:
    pool.shutdown(wait=False, cancel_futures=True)

Total: 744,360 | pages done: 78 | pages left: 74,358
12/74,358 pages | 1.8 rec/s | ETA 116.6h   
Stopped: . Rerun to resume.
